# 01 · Data Collection
**FTTH Rollout Optimizer — WBS Capstone**

This notebook handles all data ingestion:
- Synthetic dataset generation (runs immediately, no API keys needed)
- Real data download stubs for Breitbandatlas, Destatis, OSMnx
- Output: `data/processed/gemeinden_raw.csv` + `gemeinden_raw.gpkg`

To switch from synthetic → real data, follow the `# REAL DATA` blocks below.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import requests
import warnings
warnings.filterwarnings('ignore')

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Paths OK')
print(f'  raw      → {RAW.resolve()}')
print(f'  processed→ {PROCESSED.resolve()}')

## 1 · Synthetic Dataset
500 simulated Gemeinden in Bavaria. Feature distributions are calibrated to real Bavarian municipal statistics.
Swap this section with real data when available.

In [ ]:
N = 500

# Bavaria bounding box (approx): lat 47.3–50.6, lon 9.9–13.9
lats = np.random.uniform(47.3, 50.6, N)
lons = np.random.uniform(9.9,  13.9, N)

# --- Demographic features (Destatis)
population       = np.random.lognormal(mean=7.5, sigma=1.2, size=N).astype(int).clip(200, 80000)
pop_density      = np.random.lognormal(mean=4.5, sigma=1.0, size=N).clip(20, 3000)          # inhabitants/km²
avg_household_income = np.random.normal(38000, 8000, N).clip(22000, 70000)                   # €/year
share_over_65    = np.random.beta(3, 7, N) * 40                                              # %
share_under_18   = np.random.beta(4, 8, N) * 30                                              # %
unemployment_rate= np.random.beta(2, 20, N) * 10                                             # %

# --- Infrastructure features (Breitbandatlas)
existing_coverage_pct = np.random.beta(4, 2, N) * 100                                        # % homes with ≥100 Mbit/s
dist_to_pop_node_m    = np.random.lognormal(mean=6.0, sigma=0.8, size=N).clip(50, 8000)      # m to nearest POP
dist_to_cabinet_m     = np.random.lognormal(mean=5.2, sigma=0.7, size=N).clip(30, 2000)      # m to Verteilerkasten
homes_passed          = (population * np.random.uniform(0.3, 0.45, N)).astype(int)

# --- Geospatial features (OSMnx / BayernAtlas)
building_density      = pop_density * np.random.uniform(0.2, 0.5, N)                         # buildings/km²
terrain_slope_deg     = np.random.lognormal(mean=1.5, sigma=0.8, size=N).clip(0, 35)         # °
road_length_km        = np.random.lognormal(mean=4.0, sigma=0.7, size=N).clip(1, 200)        # total road km

# --- Competition
competitor_coverage   = np.random.beta(2, 3, N) * 80                                         # % covered by competitor

# --- Target: FTTH adoption rate (%)
# Simulated as a function of key drivers + noise
adoption_rate = (
    0.25 * (pop_density / pop_density.max()) +
    0.20 * (avg_household_income / avg_household_income.max()) +
    0.15 * (1 - dist_to_pop_node_m / dist_to_pop_node_m.max()) +
    0.12 * (building_density / building_density.max()) +
    0.10 * (1 - existing_coverage_pct / 100) +
    0.08 * (1 - competitor_coverage / 80) +
    0.10 * np.random.normal(0, 0.05, N)
) * 100
adoption_rate = adoption_rate.clip(5, 85)

gemeinde_names = [f'Gemeinde_{i:04d}' for i in range(N)]

df = pd.DataFrame({
    'gemeinde_id'           : range(N),
    'name'                  : gemeinde_names,
    'lat'                   : lats,
    'lon'                   : lons,
    # Demographic
    'population'            : population,
    'pop_density_km2'       : pop_density.round(1),
    'avg_household_income'  : avg_household_income.round(0).astype(int),
    'share_over_65_pct'     : share_over_65.round(1),
    'share_under_18_pct'    : share_under_18.round(1),
    'unemployment_rate_pct' : unemployment_rate.round(2),
    # Infrastructure
    'existing_coverage_pct' : existing_coverage_pct.round(1),
    'dist_to_pop_node_m'    : dist_to_pop_node_m.round(0).astype(int),
    'dist_to_cabinet_m'     : dist_to_cabinet_m.round(0).astype(int),
    'homes_passed'          : homes_passed,
    # Geospatial
    'building_density'      : building_density.round(1),
    'terrain_slope_deg'     : terrain_slope_deg.round(2),
    'road_length_km'        : road_length_km.round(2),
    # Competition
    'competitor_coverage_pct': competitor_coverage.round(1),
    # Target
    'adoption_rate_pct'     : adoption_rate.round(2),
})

print(f'Dataset shape: {df.shape}')
df.head(3)

## 2 · Convert to GeoDataFrame

In [ ]:
geometry = [Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Also reproject to UTM 32N (standard for Bavaria)
gdf_utm = gdf.to_crs('EPSG:25832')

print(f'GeoDataFrame CRS : {gdf.crs}')
print(f'UTM 32N CRS      : {gdf_utm.crs}')
gdf.head(2)

## 3 · Save Outputs

In [ ]:
csv_path  = PROCESSED / 'gemeinden_raw.csv'
gpkg_path = PROCESSED / 'gemeinden_raw.gpkg'

df.to_csv(csv_path, index=False)
gdf.to_file(gpkg_path, driver='GPKG')

print(f'CSV  saved → {csv_path}')
print(f'GPKG saved → {gpkg_path}')
print(f'Rows: {len(df)} · Columns: {len(df.columns)}')

## 4 · Quick Sanity Check

In [ ]:
print('=== Dataset Summary ===')
print(df.describe().T[['mean','std','min','max']].round(2).to_string())
print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum()>0]}')
print(f'\nTarget distribution:')
print(df['adoption_rate_pct'].describe().round(2))

---
## REAL DATA — Stubs
Uncomment and configure when real data is available.

### Breitbandatlas (Bundesnetzagentur)
Download from: https://gigabitgrundbuch.bund.de

In [ ]:
# # REAL DATA — Breitbandatlas
# breitband = gpd.read_file(RAW / 'breitbandatlas_2023.gpkg')
# breitband = breitband[breitband['bundesland'] == 'Bayern']
# breitband = breitband.to_crs('EPSG:4326')
# print(breitband.columns.tolist())

In [ ]:
# # REAL DATA — Destatis Gemeindedaten
# destatis = pd.read_excel(
#     RAW / 'destatis_gemeinden_2022.xlsx',
#     sheet_name='Gemeinden',
#     skiprows=5
# )
# destatis = destatis[destatis['Bundesland'] == 'Bayern'].copy()
# print(destatis.shape)

In [ ]:
# # REAL DATA — OSMnx building footprints
# import osmnx as ox
# place = 'Landsberg am Lech, Bavaria, Germany'
# buildings = ox.geometries_from_place(place, tags={'building': True})
# print(f'Buildings found: {len(buildings)}')
# buildings.to_file(RAW / 'buildings_landsberg.gpkg', driver='GPKG')

---
## Output Summary

| File | Description |
|------|-------------|
| `data/processed/gemeinden_raw.csv` | Flat table, 500 rows × 19 columns |
| `data/processed/gemeinden_raw.gpkg` | GeoPackage with Point geometry (EPSG:4326) |

**Next:** `02_eda_and_cleaning.ipynb`